In [1]:
# ==========================================================================================
# CELL TỔNG HỢP: CÀI ĐẶT - XỬ LÝ DỮ LIỆU (SAHI v10 - FIX LỖI ZERO DIVISION) - TRAIN - BÁO CÁO
# ==========================================================================================

import os
import glob
import shutil
import yaml
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from IPython.display import clear_output

# --- PHẦN 1: CÀI ĐẶT MÔI TRƯỜNG ---
print("🚀 [1/5] Đang cài đặt môi trường (Numpy, Ultralytics, SAHI)...")
try:
    import ultralytics
    import sahi
    print("   -> Thư viện đã được cài đặt. Bỏ qua bước pip install.")
except ImportError:
    # Gỡ các bản cũ gây xung đột
    os.system('pip uninstall -y numpy matplotlib')
    # Cài bản mới tương thích
    os.system('pip install --quiet "numpy<2.0" ultralytics "sahi[yolov5]" opencv-python-headless matplotlib seaborn')
    clear_output()
    print("✅ Cài đặt môi trường hoàn tất!")

# Import lại sau khi cài đặt
from ultralytics import YOLO

# --- CẤU HÌNH CHUNG ---
ROOT_INPUT_DIR = '/kaggle/input/dataset-seg-bien-bao/Dataset_seg_bien_bao'
OUTPUT_DIR_ORIGINAL = '/kaggle/working/yolo_dataset_original'
OUTPUT_DIR_SLICED = '/kaggle/working/yolo_dataset_sliced'
PROJECT_NAME = 'yolo_training_runs'
RUN_NAME = 'run_YOLO11_sahi_v10_final'

# --- PHẦN 2: LẬP KẾ HOẠCH & CHUẨN BỊ DỮ LIỆU ---
print("\n📦 [2/5] Đang chuẩn bị dữ liệu gốc...")
IMAGE_DIRS = [os.path.join(ROOT_INPUT_DIR, d) for d in ['seg_bien_bao_1', 'seg_bien_bao_2']]
LABEL_DIRS = [os.path.join(ROOT_INPUT_DIR, d) for d in ['labels_seg_bien_bao_1', 'labels_seg_bien_bao_2']]

# 2.1 Ghép cặp
all_original_image_paths = []
for img_dir in IMAGE_DIRS:
    all_original_image_paths.extend(glob.glob(os.path.join(img_dir, '*.jpg')))
    all_original_image_paths.extend(glob.glob(os.path.join(img_dir, '*.png')))
all_original_image_paths = sorted(all_original_image_paths)

data_pairs = []
for img_path in all_original_image_paths:
    fname = os.path.basename(img_path)
    label_fname = os.path.splitext(fname)[0] + '.txt'
    label_path = None
    for lbl_dir in LABEL_DIRS:
        potential = os.path.join(lbl_dir, label_fname)
        if os.path.exists(potential):
            label_path = potential; break
    if label_path: data_pairs.append({'original': img_path, 'label': label_path})

# 2.2 Chia tập Train/Val/Test
train_files, temp = train_test_split(data_pairs, train_size=0.7, random_state=32, shuffle=True)
val_files, test_files = train_test_split(temp, train_size=0.5, random_state=32, shuffle=True)
split_plan = {'train': train_files, 'val': val_files, 'test': test_files}

# 2.3 Copy vào thư mục chuẩn
if os.path.exists(OUTPUT_DIR_ORIGINAL): shutil.rmtree(OUTPUT_DIR_ORIGINAL)
for split, files in split_plan.items():
    os.makedirs(os.path.join(OUTPUT_DIR_ORIGINAL, f'images/{split}'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR_ORIGINAL, f'labels/{split}'), exist_ok=True)
    for item in files:
        shutil.copy(item['original'], os.path.join(OUTPUT_DIR_ORIGINAL, f'images/{split}'))
        shutil.copy(item['label'], os.path.join(OUTPUT_DIR_ORIGINAL, f'labels/{split}'))

# --- PHẦN 3: CẮT ẢNH (SAHI SLICING - V10 FIX ERROR) ---
print("\n🔪 [3/5] Đang thực hiện Slicing (V10 - Đã vá lỗi chia cho 0)...")
if os.path.exists(OUTPUT_DIR_SLICED): shutil.rmtree(OUTPUT_DIR_SLICED)

SLICE_SIZE = 640
OVERLAP = 0.2
MIN_SLICE_SIZE = 100
MIN_BOX_AREA_RATIO = 0.1 

for split in ['train', 'val', 'test']:
    img_in = os.path.join(OUTPUT_DIR_ORIGINAL, f'images/{split}')
    lbl_in = os.path.join(OUTPUT_DIR_ORIGINAL, f'labels/{split}')
    img_out = os.path.join(OUTPUT_DIR_SLICED, f'images/{split}')
    lbl_out = os.path.join(OUTPUT_DIR_SLICED, f'labels/{split}')
    os.makedirs(img_out, exist_ok=True); os.makedirs(lbl_out, exist_ok=True)
    
    # Lấy danh sách ảnh
    image_list = [f for f in os.listdir(img_in) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    for img_name in tqdm(image_list, desc=f"Slicing {split}"):
        img_path = os.path.join(img_in, img_name)
        lbl_path = os.path.join(lbl_in, os.path.splitext(img_name)[0] + '.txt')
        
        image = cv2.imread(img_path)
        if image is None: continue
        h, w = image.shape[:2]
        
        boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                boxes = [list(map(float, line.strip().split())) for line in f if len(line.strip().split()) == 5]
        
        stride_h, stride_w = int(SLICE_SIZE*(1-OVERLAP)), int(SLICE_SIZE*(1-OVERLAP))
        idx = 0
        for y in range(0, h, stride_h):
            for x in range(0, w, stride_w):
                y2, x2 = min(y + SLICE_SIZE, h), min(x + SLICE_SIZE, w)
                if (y2-y) < MIN_SLICE_SIZE or (x2-x) < MIN_SLICE_SIZE: continue
                
                # Xử lý boxes
                slice_boxes = []
                for cls, xc, yc, bw, bh in boxes:
                    bx, by, b_w, b_h = xc*w, yc*h, bw*w, bh*h
                    
                    # --- [FIX QUAN TRỌNG] Kiểm tra box lỗi ---
                    if b_w <= 0 or b_h <= 0: continue 
                    # ----------------------------------------

                    x1, y1, x2_b, y2_b = bx-b_w/2, by-b_h/2, bx+b_w/2, by+b_h/2
                    
                    if x2_b > x and x1 < x2 and y2_b > y and y1 < y2: # Giao nhau
                        nx1, ny1 = max(x1, x)-x, max(y1, y)-y
                        nx2, ny2 = min(x2_b, x2)-x, min(y2_b, y2)-y
                        
                        # Tính tỷ lệ diện tích
                        intersection_area = (nx2-nx1)*(ny2-ny1)
                        original_area = b_w*b_h
                        
                        if (intersection_area / original_area) >= MIN_BOX_AREA_RATIO:
                            sh, sw = y2-y, x2-x
                            slice_boxes.append([cls, ((nx1+nx2)/2)/sw, ((ny1+ny2)/2)/sh, (nx2-nx1)/sw, (ny2-ny1)/sh])
                
                if slice_boxes:
                    cv2.imwrite(os.path.join(img_out, f"{os.path.splitext(img_name)[0]}_s{idx}.jpg"), image[y:y2, x:x2])
                    with open(os.path.join(lbl_out, f"{os.path.splitext(img_name)[0]}_s{idx}.txt"), 'w') as f:
                        for b in slice_boxes: f.write(f"{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")
                    idx += 1

# Tạo file data.yaml
yaml_content = {'path': OUTPUT_DIR_SLICED, 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 1, 'names': ['traffic_sign']}
with open(os.path.join(OUTPUT_DIR_SLICED, 'data.yaml'), 'w') as f: yaml.dump(yaml_content, f)

# --- PHẦN 4: HUẤN LUYỆN MODEL (YOLO11 - Config Vàng V8) ---
print("\n🔥 [4/5] Bắt đầu huấn luyện YOLO11m (Có Early Stopping)...")
model = YOLO('yolo11m.pt')

results = model.train(
    data=os.path.join(OUTPUT_DIR_SLICED, 'data.yaml'),
    imgsz=640,
    epochs=320,
    patience=32,  # ✅ Early Stopping
    batch=16,
    project=PROJECT_NAME,
    name=RUN_NAME,
    device=[0, 1] if os.path.exists('/dev/nvidia1') else 0,
    exist_ok=True,
    
    # Config tối ưu
    augment=True, mosaic=0.3, 
    copy_paste=0.0, dropout=0.2, 
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    verbose=True
)

# --- PHẦN 5: TẠO BIỂU ĐỒ BÁO CÁO (FINAL) ---
print("\n📊 [5/5] Đang vẽ biểu đồ báo cáo (Chuẩn format Word)...")

def plot_report_charts(csv_path, save_dir):
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
    except Exception as e:
        print(f"❌ Không đọc được file results.csv: {e}")
        return

    sns.set(style="whitegrid")
    
    # Chart 1: Loss
    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', linewidth=2, color='blue')
    plt.plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', linewidth=2, color='orange', linestyle='--')
    plt.plot(df['epoch'], df['train/cls_loss'], label='Train Cls Loss', linewidth=2, color='green')
    plt.plot(df['epoch'], df['val/cls_loss'], label='Val Cls Loss', linewidth=2, color='red', linestyle='--')
    plt.title('Training & Validation Loss', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'BaoCao_Loss_Chart.png'), dpi=300)
    plt.close()

    # Chart 2: mAP
    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', linewidth=2.5, color='purple')
    plt.plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', linewidth=2.5, color='teal')
    plt.title('Model Performance (mAP)', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs'); plt.ylabel('Score'); plt.legend(); plt.grid(True, linestyle='--', alpha=0.7); plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'BaoCao_mAP_Chart.png'), dpi=300)
    plt.close()
    
    print(f"✅ Đã lưu 2 biểu đồ tại: {save_dir}")

# Gọi hàm vẽ
result_dir = os.path.join(PROJECT_NAME, RUN_NAME)
csv_file = os.path.join(result_dir, 'results.csv')
plot_report_charts(csv_file, result_dir)

print("\n" + "="*60)
print("🏁 HOÀN TẤT TOÀN BỘ QUY TRÌNH!")
print(f"📂 Model saved at: {os.path.join(result_dir, 'weights/best.pt')}")
print("="*60)

✅ Cài đặt môi trường hoàn tất!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

📦 [2/5] Đang chuẩn bị dữ liệu gốc...

🔪 [3/5] Đang thực hiện Slicing (V10 - Đã vá lỗi chia cho 0)...


Slicing train:   0%|          | 0/1407 [00:00<?, ?it/s]

Slicing val:   0%|          | 0/301 [00:00<?, ?it/s]

Slicing test:   0%|          | 0/302 [00:00<?, ?it/s]


🔥 [4/5] Bắt đầu huấn luyện YOLO11m (Có Early Stopping)...
Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset_sliced/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, epochs=320, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.3, mult

In [2]:
# # ===================================================================
# # CELL 1: CÀI ĐẶT VÀ THIẾT LẬP MÔI TRƯỜNG
# # >> ĐẶT CELL NÀY Ở ĐẦU TIÊN TRONG NOTEBOOK CỦA BẠN <<
# # ===================================================================
# # Lệnh này sẽ được chạy đầu tiên khi bạn nhấn "Save & Run All",
# # đảm bảo môi trường được cài đặt đúng trước khi các cell khác chạy.

# print("--- Bắt đầu cài đặt môi trường ---")

# # 1. Gỡ cài đặt các phiên bản có thể gây xung đột (để đảm bảo)
# !pip uninstall -y numpy matplotlib

# # 2. Cài đặt các phiên bản tương thích và thư viện ultralytics
# #    - numpy<2.0: Để tránh xung đột với matplotlib và các thư viện khác.
# #    - ultralytics: Thư viện chính để huấn luyện YOLO.
# !pip install --quiet "numpy<2.0" ultralytics

# print("\n--- Cài đặt môi trường hoàn tất ---")
# # Sau khi cell này chạy xong, môi trường đã sẵn sàng cho các bước tiếp theo.

TEST PREPROCESSED DATASET

In [3]:
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# import os
# import random
# import glob

# # --- 1. CẤU HÌNH ---
# # Điều chỉnh đường dẫn này cho phù hợp với notebook Kaggle của bạn
# # Dựa trên ảnh của bạn, nó sẽ chứa 2 thư mục con là seg_bien_bao_1 và seg_bien_bao_2
# DATASET_DIR = '/kaggle/input/dataset-seg-bien-bao/Dataset_seg_bien_bao'
# IMAGE_DIRS = [
#     os.path.join(DATASET_DIR, 'seg_bien_bao_1'),
#     os.path.join(DATASET_DIR, 'seg_bien_bao_2')
# ]

# # Số lượng ảnh mẫu để hiển thị
# NUM_SAMPLES_TO_SHOW = 5

# # --- 2. HÀM TIỀN XỬ LÝ ẢNH ---
# def preprocess_image_for_yolo(image):
#     """
#     Áp dụng chuỗi các bước tiền xử lý để cải thiện chất lượng ảnh cho mô hình YOLO.
#     """
#     # Bước 1: Chuyển ảnh từ BGR (mặc định của OpenCV) sang LAB
#     lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

#     # Tách các kênh L, A, B
#     l_channel, a_channel, b_channel = cv2.split(lab_image)

#     # Bước 2: Áp dụng CLAHE cho kênh L (Luminance)
#     # clipLimit: Ngưỡng giới hạn độ tương phản. Giá trị 2.0-3.0 thường tốt.
#     # tileGridSize: Kích thước của các vùng nhỏ để áp dụng cân bằng. (8, 8) là giá trị phổ biến.
#     clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
#     clahe_l_channel = clahe.apply(l_channel)

#     # Bước 3: Ghép kênh L đã xử lý với các kênh A, B gốc
#     merged_lab_image = cv2.merge([clahe_l_channel, a_channel, b_channel])

#     # Chuyển ngược từ LAB về BGR
#     enhanced_image = cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2BGR)
    
#     # Bước 4 (Tùy chọn): Tăng độ sắc nét cho ảnh
#     # Tạo một sharpening kernel
#     sharpen_kernel = np.array([
#         [-1, -1, -1],
#         [-1,  9, -1],
#         [-1, -1, -1]
#     ])
#     # Áp dụng kernel lên ảnh đã được cải thiện
#     sharpened_image = cv2.filter2D(enhanced_image, -1, sharpen_kernel)

#     return sharpened_image

# # --- 3. TẢI VÀ TRỰC QUAN HÓA KẾT QUẢ ---
# # Lấy tất cả các đường dẫn ảnh từ các thư mục con
# all_image_paths = []
# for img_dir in IMAGE_DIRS:
#     # Sử dụng glob để tìm tất cả các file ảnh (ví dụ: .jpg, .png)
#     all_image_paths.extend(glob.glob(os.path.join(img_dir, '*.jpg')))
#     all_image_paths.extend(glob.glob(os.path.join(img_dir, '*.png')))
#     all_image_paths.extend(glob.glob(os.path.join(img_dir, '*.jpeg')))


# # Chọn ngẫu nhiên một vài ảnh để xem thử
# if len(all_image_paths) > NUM_SAMPLES_TO_SHOW:
#     sample_paths = random.sample(all_image_paths, NUM_SAMPLES_TO_SHOW)
# else:
#     sample_paths = all_image_paths

# print(f"Tìm thấy tổng cộng {len(all_image_paths)} ảnh.")
# print(f"Hiển thị {len(sample_paths)} ảnh mẫu sau khi tiền xử lý...")

# # Vẽ kết quả
# for image_path in sample_paths:
#     # Đọc ảnh gốc bằng OpenCV
#     original_image = cv2.imread(image_path)
    
#     if original_image is None:
#         print(f"Không thể đọc ảnh: {image_path}")
#         continue

#     # Tiền xử lý ảnh
#     preprocessed_image = preprocess_image_for_yolo(original_image)

#     # Chuyển ảnh từ BGR sang RGB để matplotlib hiển thị đúng màu
#     original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
#     preprocessed_image_rgb = cv2.cvtColor(preprocessed_image, cv2.COLOR_BGR2RGB)

#     # Hiển thị
#     fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    
#     axes[0].imshow(original_image_rgb)
#     axes[0].set_title('Ảnh Gốc')
#     axes[0].axis('off')

#     axes[1].imshow(preprocessed_image_rgb)
#     axes[1].set_title('Ảnh đã Tiền xử lý (CLAHE + Sharpening)')
#     axes[1].axis('off')

#     plt.suptitle(f"Ảnh: {os.path.basename(image_path)}", fontsize=16)
#     plt.tight_layout(rect=[0, 0, 1, 0.96])
#     plt.show()

TRAIN TEST SPLIT

In [4]:
# # ===================================================================
# # CELL 2: LẬP KẾ HOẠCH PHÂN CHIA DỮ LIỆU DÙNG CHUNG
# # >> CHẠY CELL NÀY SAU KHI CÀI ĐẶT MÔI TRƯỜNG <<
# # ===================================================================

# import os
# import glob
# from sklearn.model_selection import train_test_split
# from tqdm import tqdm

# print("--- Bắt đầu lập kế hoạch phân chia dữ liệu ---")

# # --- 1. CẤU HÌNH ---
# ROOT_INPUT_DIR = '/kaggle/input/dataset-seg-bien-bao/Dataset_seg_bien_bao'
# IMAGE_DIRS = [os.path.join(ROOT_INPUT_DIR, d) for d in ['seg_bien_bao_1', 'seg_bien_bao_2']]
# LABEL_DIRS = [os.path.join(ROOT_INPUT_DIR, d) for d in ['labels_seg_bien_bao_1', 'labels_seg_bien_bao_2']]
# TRAIN_RATIO = 0.7
# VAL_RATIO = 0.15

# # --- 2. TÌM VÀ GHÉP CẶP TẤT CẢ ẢNH GỐC VÀ NHÃN ---
# all_original_image_paths = []
# for img_dir in IMAGE_DIRS:
#     all_original_image_paths.extend(glob.glob(os.path.join(img_dir, '*.jpg')))
#     all_original_image_paths.extend(glob.glob(os.path.join(img_dir, '*.png')))
# all_original_image_paths = sorted(all_original_image_paths) # Sắp xếp để đảm bảo thứ tự

# data_pairs = []
# for img_path in tqdm(all_original_image_paths, desc="Ghép cặp ảnh-nhãn"):
#     fname = os.path.basename(img_path)
#     label_fname = os.path.splitext(fname)[0] + '.txt'
#     label_path = None
#     for lbl_dir in LABEL_DIRS:
#         potential_path = os.path.join(lbl_dir, label_fname)
#         if os.path.exists(potential_path):
#             label_path = potential_path
#             break
#     if label_path:
#         # Lưu đường dẫn ảnh gốc và nhãn tương ứng
#         data_pairs.append({'original': img_path, 'label': label_path})

# # --- 3. THỰC HIỆN PHÂN CHIA MỘT LẦN DUY NHẤT ---
# train_files, temp_files = train_test_split(data_pairs, train_size=TRAIN_RATIO, random_state=32, shuffle=True)
# val_files, test_files = train_test_split(temp_files, train_size=0.5, random_state=32, shuffle=True)

# # Lưu kế hoạch vào một biến để các cell sau sử dụng
# split_plan = {'train': train_files, 'val': val_files, 'test': test_files}

# print("\n--- Hoàn tất lập kế hoạch phân chia dữ liệu ---")
# print(f"Tổng số cặp ảnh-nhãn hợp lệ: {len(data_pairs)}")
# print(f" - Tập Train: {len(split_plan['train'])} ảnh")
# print(f" - Tập Validation: {len(split_plan['val'])} ảnh")
# print(f" - Tập Test: {len(split_plan['test'])} ảnh")

TRAIN MODEL PREPROCESSED DATASET

In [5]:
# # ===================================================================
# # CELL 3: TẠO DATASET VÀ HUẤN LUYỆN TRÊN ẢNH ĐÃ TIỀN XỬ LÝ
# # >> CHẠY CELL NÀY SAU KHI ĐÃ CÓ `split_plan` TỪ CELL 2 <<
# # ===================================================================
# # Nhiệm vụ:
# # 1. Dùng `split_plan` để xác định ảnh nào thuộc tập train/val/test.
# # 2. Đọc ảnh gốc, áp dụng tiền xử lý (CLAHE + Sharpening).
# # 3. Lưu ảnh đã xử lý và nhãn tương ứng vào cấu trúc thư mục mới.
# # 4. Huấn luyện model YOLOv8 trên bộ dữ liệu đã tiền xử lý này.
# # ===================================================================

# import cv2
# import numpy as np
# import os
# import shutil
# import yaml
# from tqdm import tqdm
# from ultralytics import YOLO

# # --- 1. HÀM TIỀN XỬ LÝ ẢNH ---
# def preprocess_image_for_yolo(image):
#     """Áp dụng CLAHE và Sharpening để cải thiện chất lượng ảnh."""
#     lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
#     l_channel, a_channel, b_channel = cv2.split(lab_image)
#     clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
#     clahe_l_channel = clahe.apply(l_channel)
#     merged_lab_image = cv2.merge([clahe_l_channel, a_channel, b_channel])
#     enhanced_image = cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2BGR)
#     sharpen_kernel = np.array([[-1, -1, -1], [-1,  9, -1], [-1, -1, -1]])
#     sharpened_image = cv2.filter2D(enhanced_image, -1, sharpen_kernel)
#     return sharpened_image

# # --- 2. TẠO BỘ DỮ LIỆU ĐÃ TIỀN XỬ LÝ DỰA TRÊN `split_plan` ---
# print("--- Bắt đầu tạo bộ dữ liệu đã tiền xử lý (dựa trên kế hoạch) ---")
# OUTPUT_DIR_PREPROCESSED = '/kaggle/working/yolo_dataset_preprocessed' # Đặt tên thư mục rõ ràng

# # Xóa thư mục cũ nếu tồn tại để tránh lỗi
# if os.path.exists(OUTPUT_DIR_PREPROCESSED):
#     shutil.rmtree(OUTPUT_DIR_PREPROCESSED)

# # Vòng lặp qua các tập train, val, test trong kế hoạch
# for split_name, file_list in split_plan.items():
#     img_dir = os.path.join(OUTPUT_DIR_PREPROCESSED, f'images/{split_name}')
#     lbl_dir = os.path.join(OUTPUT_DIR_PREPROCESSED, f'labels/{split_name}')
#     os.makedirs(img_dir, exist_ok=True)
#     os.makedirs(lbl_dir, exist_ok=True)

#     for item in tqdm(file_list, desc=f"Xử lý và sao chép tập {split_name}"):
#         # Đọc ảnh gốc
#         original_image = cv2.imread(item['original'])
#         if original_image is None:
#             continue
            
#         # Tiền xử lý ảnh
#         preprocessed_image = preprocess_image_for_yolo(original_image)
        
#         # Lưu ảnh đã xử lý vào thư mục mới
#         img_savename = os.path.basename(item['original'])
#         cv2.imwrite(os.path.join(img_dir, img_savename), preprocessed_image)
        
#         # Sao chép file nhãn tương ứng
#         shutil.copy(item['label'], lbl_dir)

# print("\n--- Hoàn tất tạo bộ dữ liệu đã tiền xử lý ---")

# # --- 3. TẠO FILE YAML CHO BỘ DỮ LIỆU MỚI ---
# yaml_content = {
#     'path': OUTPUT_DIR_PREPROCESSED,
#     'train': 'images/train',
#     'val': 'images/val',
#     'test': 'images/test',
#     'nc': 1,
#     'names': ['traffic_sign']
# }
# yaml_file_path = os.path.join(OUTPUT_DIR_PREPROCESSED, 'data.yaml')
# with open(yaml_file_path, 'w') as f:
#     yaml.dump(yaml_content, f, sort_keys=False)

# print(f"File cấu hình đã được tạo tại: {yaml_file_path}")

# # --- 4. HUẤN LUYỆN MODEL TRÊN DỮ LIỆU ĐÃ TIỀN XỬ LÝ ---
# print("\n--- Bắt đầu huấn luyện model trên DỮ LIỆU ĐÃ TIỀN XỬ LÝ ---")

# # Tải mô hình pretrained
# model_preprocessed = YOLO('yolov8m.pt')

# # Bắt đầu huấn luyện
# results_preprocessed = model_preprocessed.train(
#     data=yaml_file_path,
#     imgsz=640,
#     epochs=75,
#     batch=16,
#     project='yolo_training_runs', # Tên project chung để lưu kết quả
#     name='run_preprocessed',      # Tên của lần chạy này, để phân biệt
#     device=[0, 1],
#     exist_ok=True # Cho phép ghi đè nếu chạy lại cell
# )

# print("\n--- HOÀN TẤT HUẤN LUYỆN TRÊN DỮ LIỆU ĐÃ TIỀN XỬ LÝ ---")

TRAIN MODEL NO PREPROCESSED DATASET

In [6]:
# # ===================================================================
# # CELL 4: TẠO DATASET VÀ HUẤN LUYỆN TRÊN ẢNH GỐC (KHÔNG TIỀN XỬ LÝ)
# # >> CHẠY CELL NÀY SAU KHI ĐÃ HUẤN LUYỆN MODEL TIỀN XỬ LÝ <<
# # ===================================================================
# # Nhiệm vụ:
# # 1. Dùng lại `split_plan` từ Cell 2.
# # 2. Sao chép ảnh GỐC và nhãn vào một cấu trúc thư mục mới.
# # 3. Huấn luyện model YOLOv8 thứ hai trên bộ dữ liệu gốc này.
# # ===================================================================

# import os
# import shutil
# import yaml
# from tqdm import tqdm
# from ultralytics import YOLO

# # --- 1. TẠO BỘ DỮ LIỆU ẢNH GỐC DỰA TRÊN `split_plan` ---
# print("--- Bắt đầu tạo bộ dữ liệu ảnh gốc (dựa trên kế hoạch) ---")
# OUTPUT_DIR_ORIGINAL = '/kaggle/working/yolo_dataset_original' # Đặt tên thư mục rõ ràng

# # Xóa thư mục cũ nếu tồn tại để tránh lỗi
# if os.path.exists(OUTPUT_DIR_ORIGINAL):
#     shutil.rmtree(OUTPUT_DIR_ORIGINAL)

# # Vòng lặp qua các tập train, val, test trong kế hoạch
# for split_name, file_list in split_plan.items():
#     img_dir = os.path.join(OUTPUT_DIR_ORIGINAL, f'images/{split_name}')
#     lbl_dir = os.path.join(OUTPUT_DIR_ORIGINAL, f'labels/{split_name}')
#     os.makedirs(img_dir, exist_ok=True)
#     os.makedirs(lbl_dir, exist_ok=True)

#     for item in tqdm(file_list, desc=f"Sao chép tập {split_name}"):
#         # Chỉ sao chép ảnh gốc, không xử lý gì cả
#         shutil.copy(item['original'], img_dir)
        
#         # Sao chép file nhãn tương ứng
#         shutil.copy(item['label'], lbl_dir)

# print("\n--- Hoàn tất tạo bộ dữ liệu ảnh gốc ---")

# # --- 2. TẠO FILE YAML CHO BỘ DỮ LIỆU ẢNH GỐC ---
# yaml_content_original = {
#     'path': OUTPUT_DIR_ORIGINAL,
#     'train': 'images/train',
#     'val': 'images/val',
#     'test': 'images/test',
#     'nc': 1,
#     'names': ['traffic_sign']
# }
# yaml_file_path_original = os.path.join(OUTPUT_DIR_ORIGINAL, 'data.yaml')
# with open(yaml_file_path_original, 'w') as f:
#     yaml.dump(yaml_content_original, f, sort_keys=False)

# print(f"File cấu hình đã được tạo tại: {yaml_file_path_original}")

# # --- 3. HUẤN LUYỆN MODEL TRÊN DỮ LIỆU GỐC ---
# print("\n--- Bắt đầu huấn luyện model trên DỮ LIỆU GỐC ---")

# # Tải mô hình pretrained
# model_original = YOLO('yolov8m.pt')

# # Bắt đầu huấn luyện
# results_original = model_original.train(
#     data=yaml_file_path_original,
#     imgsz=640,
#     epochs=75,
#     batch=16,
#     project='yolo_training_runs', # Lưu vào cùng project chung
#     name='run_original',          # Tên của lần chạy này, để phân biệt
#     device=[0, 1],
#     exist_ok=True # Cho phép ghi đè nếu chạy lại cell
# )

# print("\n--- HOÀN TẤT HUẤN LUYỆN TRÊN DỮ LIỆU GỐC ---")

COMPARE 2 MODEL

In [7]:
# # ===================================================================
# # CELL 5: SO SÁNH TOÀN DIỆN VÀ ĐÁNH GIÁ CUỐI CÙNG
# # >> ĐẶT CELL NÀY Ở CUỐI CÙNG TRONG NOTEBOOK <<
# # ===================================================================
# # Nhiệm vụ:
# # 1. Tải 2 model 'best.pt' từ 2 lần huấn luyện.
# # 2. Lấy ngẫu nhiên 5 ảnh từ tập test chung.
# # 3. Chạy dự đoán và hiển thị kết quả của 2 model cạnh nhau.
# # 4. Vẽ biểu đồ so sánh mAP50 qua các epoch từ 2 lần huấn luyện.
# # ===================================================================

# import os
# import random
# import cv2
# import pandas as pd
# import matplotlib.pyplot as plt
# from ultralytics import YOLO
# from IPython.display import display

# print("--- Bắt đầu quy trình so sánh và đánh giá cuối cùng ---")

# # --- 1. CẤU HÌNH VÀ TẢI CÁC MODEL ĐÃ HUẤN LUYỆN ---
# # Đường dẫn phải khớp với tham số 'project' và 'name' bạn đã đặt
# path_original_run = '/kaggle/working/yolo_training_runs/run_original'
# path_preprocessed_run = '/kaggle/working/yolo_training_runs/run_preprocessed'

# path_model_original = os.path.join(path_original_run, 'weights/best.pt')
# path_model_preprocessed = os.path.join(path_preprocessed_run, 'weights/best.pt')

# model_original = None
# model_preprocessed = None

# if os.path.exists(path_model_original):
#     model_original = YOLO(path_model_original)
#     print("Tải thành công model huấn luyện trên 'Ảnh Gốc'.")
# else:
#     print(f"LỖI: Không tìm thấy model 'Ảnh Gốc' tại: {path_model_original}")

# if os.path.exists(path_model_preprocessed):
#     model_preprocessed = YOLO(path_model_preprocessed)
#     print("Tải thành công model huấn luyện trên 'Ảnh Tiền xử lý'.")
# else:
#     print(f"LỖI: Không tìm thấy model 'Ảnh Tiền xử lý' tại: {path_model_preprocessed}")

# # --- 2. HÀM TIỀN XỬ LÝ (Để tái sử dụng) ---
# def preprocess_image_for_yolo(image):
#     """Áp dụng CLAHE và Sharpening để cải thiện chất lượng ảnh."""
#     lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
#     l_channel, a_channel, b_channel = cv2.split(lab_image)
#     clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
#     clahe_l_channel = clahe.apply(l_channel)
#     merged_lab_image = cv2.merge([clahe_l_channel, a_channel, b_channel])
#     enhanced_image = cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2BGR)
#     sharpen_kernel = np.array([[-1, -1, -1], [-1,  9, -1], [-1, -1, -1]])
#     sharpened_image = cv2.filter2D(enhanced_image, -1, sharpen_kernel)
#     return sharpened_image

# # --- 3. SO SÁNH TRỰC QUAN TRÊN ẢNH TEST NGẪU NHIÊN ---
# if model_original and model_preprocessed and 'split_plan' in locals():
#     print("\n--- Bắt đầu so sánh trực quan kết quả dự đoán ---")
    
#     # Lấy 5 ảnh ngẫu nhiên từ tập test đã được chia ở Cell 2
#     num_samples = 5
#     test_image_list = split_plan['test']
#     if len(test_image_list) >= num_samples:
#         sample_items = random.sample(test_image_list, num_samples)

#         for item in sample_items:
#             path_original_img = item['original']
            
#             # Đọc và tiền xử lý ảnh
#             original_img_cv2 = cv2.imread(path_original_img)
#             preprocessed_img_cv2 = preprocess_image_for_yolo(original_img_cv2)

#             # Chạy dự đoán
#             results_orig = model_original(original_img_cv2, verbose=False)
#             results_prep = model_preprocessed(preprocessed_img_cv2, verbose=False)

#             # Vẽ kết quả
#             img_res_orig = results_orig[0].plot() # Trả về ảnh BGR
#             img_res_prep = results_prep[0].plot() # Trả về ảnh BGR
            
#             # Chuyển sang RGB để hiển thị
#             img_res_orig_rgb = cv2.cvtColor(img_res_orig, cv2.COLOR_BGR2RGB)
#             img_res_prep_rgb = cv2.cvtColor(img_res_prep, cv2.COLOR_BGR2RGB)

#             # Hiển thị cạnh nhau
#             fig, axes = plt.subplots(1, 2, figsize=(20, 10))
#             axes[0].imshow(img_res_orig_rgb)
#             axes[0].set_title('Dự đoán trên Ảnh Gốc', fontsize=14)
#             axes[0].axis('off')

#             axes[1].imshow(img_res_prep_rgb)
#             axes[1].set_title('Dự đoán trên Ảnh Tiền xử lý', fontsize=14)
#             axes[1].axis('off')
            
#             plt.suptitle(f"So sánh trên ảnh: {os.path.basename(path_original_img)}", fontsize=16)
#             plt.show()
#     else:
#         print("Không đủ ảnh trong tập test để lấy mẫu.")
# else:
#     print("\nBỏ qua so sánh trực quan do không tải được model hoặc thiếu 'split_plan'.")


# # --- 4. VẼ BIỂU ĐỒ SO SÁNH HIỆU SUẤT HUẤN LUYỆN ---
# print("\n--- Bắt đầu vẽ biểu đồ so sánh hiệu suất huấn luyện ---")

# path_original_csv = os.path.join(path_original_run, 'results.csv')
# path_preprocessed_csv = os.path.join(path_preprocessed_run, 'results.csv')

# df_original = None
# df_preprocessed = None

# if os.path.exists(path_original_csv):
#     df_original = pd.read_csv(path_original_csv)
#     df_original.columns = df_original.columns.str.strip()
#     print("Đã tải thành công file kết quả của model 'Ảnh Gốc'.")
# else:
#     print(f"CẢNH BÁO: Không tìm thấy file kết quả cho model 'Ảnh Gốc' tại: {path_original_csv}")

# if os.path.exists(path_preprocessed_csv):
#     df_preprocessed = pd.read_csv(path_preprocessed_csv)
#     df_preprocessed.columns = df_preprocessed.columns.str.strip()
#     print("Đã tải thành công file kết quả của model 'Ảnh Tiền xử lý'.")
# else:
#     print(f"CẢNH BÁO: Không tìm thấy file kết quả cho model 'Ảnh Tiền xử lý' tại: {path_preprocessed_csv}")

# if df_original is not None or df_preprocessed is not None:
#     plt.style.use('seaborn-v0_8-whitegrid')
#     fig, ax = plt.subplots(figsize=(16, 9))

#     if df_original is not None and 'metrics/mAP50(B)' in df_original.columns:
#         ax.plot(df_original['epoch'], df_original['metrics/mAP50(B)'], marker='o', linestyle='-', label='Model train trên Ảnh Gốc')

#     if df_preprocessed is not None and 'metrics/mAP50(B)' in df_preprocessed.columns:
#         ax.plot(df_preprocessed['epoch'], df_preprocessed['metrics/mAP50(B)'], marker='s', linestyle='--', label='Model train trên Ảnh Tiền xử lý')

#     ax.set_title('So sánh Hiệu suất Huấn luyện giữa Ảnh Gốc và Ảnh Tiền xử lý', fontsize=18)
#     ax.set_xlabel('Epoch', fontsize=14)
#     ax.set_ylabel('mAP50 (Validation)', fontsize=14)
#     ax.legend(fontsize=12)
#     ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    
#     # Chú thích điểm cao nhất
#     if df_original is not None and 'metrics/mAP50(B)' in df_original.columns:
#         best_epoch_orig = df_original['metrics/mAP50(B)'].idxmax()
#         best_map_orig = df_original['metrics/mAP50(B)'].max()
#         ax.annotate(f'Max: {best_map_orig:.3f}', xy=(best_epoch_orig, best_map_orig), xytext=(best_epoch_orig, best_map_orig + 0.02),
#                     arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8), ha='center', fontsize=10)

#     if df_preprocessed is not None and 'metrics/mAP50(B)' in df_preprocessed.columns:
#         best_epoch_prep = df_preprocessed['metrics/mAP50(B)'].idxmax()
#         best_map_prep = df_preprocessed['metrics/mAP50(B)'].max()
#         ax.annotate(f'Max: {best_map_prep:.3f}', xy=(best_epoch_prep, best_map_prep), xytext=(best_epoch_prep, best_map_prep - 0.04),
#                     arrowprops=dict(facecolor='blue', shrink=0.05, width=1, headwidth=8), ha='center', fontsize=10)

#     plt.tight_layout()
#     plt.show()
# else:
#     print("\nKhông có đủ dữ liệu để vẽ biểu đồ so sánh.")